# ITT parcial (base social) - Barrio Obrero (oficial, 45 manzanas) (Cali)

Visor del **Indice de Transformacion Territorial (ITT)** de Barrio Obrero
replicando el estandar de Ciudad Paraiso (mapa base, paleta Okabe-Ito,
jerarquia de capas, widget flotante, 4 niveles oficiales).

> **ITT PARCIAL.** De las 5 dimensiones del ITT, solo la dimension
> **Social/Vivienda** tiene dato para Barrio Obrero (derivada del IPM:
> `social = 100 - IPM`, por manzana). Las otras 4 (Seguridad, Convivencia,
> Movilidad, Entorno Urbano) se muestran rotuladas **"Sin datos para
> Comuna 9 (pendiente DATIC)"**, porque el dataset ITT real solo cubre
> Ciudad Paraiso (Comuna 3). El score mostrado es PARCIAL (solo base social).

- Fuente social: Censo 2018 (DANE, CNPV) via IPM. Vacios = 0.
- Niveles ITT: Activacion 0-40 (#D55E00), Consolidacion 40-60 (#E69F00),
  Transformacion 60-80 (#009E73), Escala 80-100 (#0072B2).
- Salida: `mapa_html/Barrio_obrero/mapa_itt_barrio_obrero_original.html`.


## 0. Setup Colab (opcional)

In [ ]:
def _en_colab():
    try:
        import google.colab
        return True
    except Exception:
        return False
if _en_colab():
    import os,subprocess,sys
    REPO='https://github.com/j0rg3c45/mapa_Analitico_ciudad_Paraiso.git'
    DEST='/content/mapa_Analitico_ciudad_Paraiso'
    subprocess.run([sys.executable,'-m','pip','install','-q','geopandas','folium','mapclassify','pyarrow','openpyxl'],check=False)
    if not os.path.isdir(DEST): subprocess.run(['git','clone','--depth','1',REPO,DEST],check=False)
    if os.path.isdir(os.path.join(DEST,'notebooks')): os.chdir(os.path.join(DEST,'notebooks'))
    print('Colab listo:',os.getcwd())
else:
    print('Entorno local (uv). Setup Colab omitido.')


## 1. Entorno y paleta

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
# Paleta Okabe-Ito. Niveles ITT (los 4 oficiales del proyecto).
OKI_AZUL="#0072B2"; OKI_NARANJA="#E69F00"; OKI_VERDE="#009E73"; OKI_BERMELL="#D55E00"
OKI_GRIS="#333333"; BG="#F4F6F9"; CRS_WEB=4326; LIMITE="#1B4F8A"
ITT_NIVELES=[(0.0,40.0,"Activacion","#D55E00"),(40.0,60.0,"Consolidacion","#E69F00"),
             (60.0,80.0,"Transformacion","#009E73"),(80.0,100.0,"Escala","#0072B2")]
def _itt_nivel(v):
    v=float(v)
    for lo,hi,nom,col in ITT_NIVELES:
        if v<=hi: return nom,col
    return ITT_NIVELES[-1][2],ITT_NIVELES[-1][3]
print("Entorno listo.")

## 2. Rutas, poligono y salida

In [ ]:
def _ruta_bo(*partes):
    for base in [os.path.join("..","data","Barrio_Obrero_geojson"),
                 os.path.join("data","Barrio_Obrero_geojson"),
                 os.path.join("/content","data","Barrio_Obrero_geojson")]:
        p=os.path.join(base,*partes)
        if os.path.isfile(p): return os.path.abspath(p)
    return None
_pp=_ruta_bo("Geojson_Barrio_Obrero.geojson")
_rp=_ruta_bo("IPM_Barrio_Obrero_completo.geojson")
if _pp is None or _rp is None:
    raise FileNotFoundError("Falta el poligono o el IPM de Barrio Obrero (oficial, 45 manzanas)")
POLIGONO_ZONA=gpd.read_file(_pp).to_crs(CRS_WEB)
_b=POLIGONO_ZONA.total_bounds
BO_BOUNDS=[[float(_b[1]),float(_b[0])],[float(_b[3]),float(_b[2])]]
OUT_DIR=None
for _d in [os.path.join("..","data_processed_geo"),"data_processed_geo","/content/data_processed_geo"]:
    try:
        os.makedirs(_d,exist_ok=True); OUT_DIR=os.path.abspath(_d); break
    except Exception: continue
MAPAS_DIR=os.path.abspath(os.path.join(os.path.dirname(OUT_DIR.rstrip(os.sep)),"mapa_html","Barrio_obrero"))
os.makedirs(MAPAS_DIR,exist_ok=True)
print("Poligono Barrio Obrero (oficial, 45 manzanas) | bounds:", BO_BOUNDS)
print("Salida de mapas HTML:", MAPAS_DIR)

## 3. Calculo del ITT parcial (dimension Social/Vivienda desde el IPM)

In [ ]:
# Carga de manzanas con IPM y 15 privaciones.
mz = gpd.read_file(_rp)
mz = mz.set_crs(epsg=CRS_WEB) if mz.crs is None else mz.to_crs(CRS_WEB)
mz = mz[mz.geometry.notna()].reset_index(drop=True)
mz["ipm"] = pd.to_numeric(mz["ipm"], errors="coerce").fillna(0.0)
# Recorte al poligono de la zona (por si el IPM trae manzanas de mas).
_z = POLIGONO_ZONA.geometry.union_all()
mz = mz[mz.geometry.representative_point().within(_z) | mz.geometry.intersects(_z)].reset_index(drop=True)
# Privaciones (para marcar sin poblacion y para el cruce en el panel).
VARS=["analf_","bajo_","infancia_","inasis_","rezago_","trab_infan_","depen_","infor_",
      "salud_","asegu_","haci_","pared_","excre_","pisos_","agua_"]
COLMAP={"trab_infan_":"TRAB_INFAN"}
for v in VARS:
    _s=COLMAP.get(v,v.upper())
    mz[v]=pd.to_numeric(mz[_s],errors="coerce").fillna(0.0) if _s in mz.columns else 0.0
mz["sin_pob"]=(mz["ipm"]==0)&(mz["bajo_"]==0)&(mz["infor_"]==0)
# Dimension Social/Vivienda del ITT: a mayor IPM (privacion) menor transformacion.
#   social = 100 - IPM (acotado 0-100). Es la UNICA dimension con dato en Comuna 9.
mz["d_social"]=(100.0-mz["ipm"]).clip(0,100)
# Las otras 4 dimensiones no tienen dato (pendiente DATIC) -> NaN.
for _d in ["d_seguridad","d_convivencia","d_movilidad","d_entorno"]:
    mz[_d]=np.nan
# Score ITT PARCIAL: con una sola dimension disponible, el parcial = social.
mz["itt_parcial"]=mz["d_social"]
mz["area_m2"]=mz.to_crs(3116).geometry.area.values
# Resumen de zona (excluye manzanas sin poblacion del promedio).
_con=mz[~mz["sin_pob"]]
ITT_ZONA=round(float(_con["itt_parcial"].mean()),1) if len(_con) else 0.0
IPM_ZONA=round(float(_con["ipm"].mean()),1) if len(_con) else 0.0
_niv_zona,_=_itt_nivel(ITT_ZONA)
print("Manzanas:",len(mz),"| con poblacion:",int((~mz['sin_pob']).sum()))
print("ITT parcial (base social) de zona: %.1f -> %s | IPM zona: %.1f" % (ITT_ZONA,_niv_zona,IPM_ZONA))

## 4. Tabla resumen del ITT parcial por manzana

In [ ]:
# Distribucion de manzanas por nivel ITT (solo con poblacion).
_dist={nom:0 for _lo,_hi,nom,_c in ITT_NIVELES}
for _v in _con["itt_parcial"]:
    _n,_=_itt_nivel(_v); _dist[_n]+=1
_tabla=pd.DataFrame({
  "Dimension":["Social / Vivienda (IPM)","Seguridad","Convivencia","Movilidad","Entorno Urbano"],
  "Estado":["CON DATO","SIN DATO","SIN DATO","SIN DATO","SIN DATO"],
  "Score (0-100)":[ITT_ZONA,None,None,None,None]})
print(_tabla.to_string(index=False))
print("\nManzanas por nivel ITT (base social):")
for _lo,_hi,nom,_c in ITT_NIVELES:
    print("  %-15s (%d-%d): %d" % (nom,int(_lo),int(_hi),_dist[nom]))

## 5. Exportacion de lamina (deck) - helper reutilizable

In [ ]:
# Helper reutilizable para inyectar en un mapa Folium UN boton de exportacion
# de lamina con estructura de deck (PMU). Captura el contenedor del mapa como
# imagen y la compone dentro de una lamina HTML (encabezado + 2 columnas +
# pie) que se exporta a PDF (una pagina). Este archivo es solo una plantilla
# de trabajo; su contenido se inserta como celda en los notebooks.
#
# Decision CORS/tiles: html2canvas no puede rasterizar los tiles base de
# Google/Esri por restricciones CORS (saldrian en blanco o rotos). Por eso,
# antes de capturar el mapa se oculta el panel de tiles base (.leaflet-tile-pane)
# y se pinta un fondo claro (#F4F6F9) en el contenedor; se capturan solo los
# overlays vectoriales (contorno, manzanas, puntos, calor). La imagen del mapa
# va en la columna derecha de la lamina.
def agregar_export(mapa, titulo, slug):
    """Inyecta en 'mapa' (folium.Map) un control flotante con UN boton:
    'Exportar lamina (deck)'. Al pulsar, captura el contenedor del mapa a una
    imagen y construye una lamina HTML con estructura de deck del PMU
    (encabezado dorado, 2 columnas: resumen/cifras clonadas del panel +
    mapa, pie con corte y fuente); captura la lamina y la exporta a PDF de
    una pagina (landscape) con dialogo "Guardar como" (showSaveFilePicker) y
    fallback a descarga. Nombre: deck_<slug>_<AAAA-MM-DD>.pdf. Debe llamarse
    ANTES de mapa.save(...).
    - titulo: titulo visible en el encabezado de la lamina.
    - slug: nombre corto para el archivo.
    """
    map_id = mapa.get_name()  # id del div .folium-map de ESTE mapa
    # Librerias por CDN (sin build).
    cdn = (
        '<script src="https://cdn.jsdelivr.net/npm/html2canvas@1.4.1/dist/html2canvas.min.js"></script>'
        '<script src="https://cdn.jsdelivr.net/npm/jspdf@2.5.1/dist/jspdf.umd.min.js"></script>'
    )
    mapa.get_root().header.add_child(folium.Element(cdn))
    # Control + logica. Se escapa como string de Python; en JS se usa map_id,
    # titulo y slug concretos de ESTE mapa (asi funciona con varios mapas).
    titulo_js = titulo.replace("\\", "\\\\").replace("'", "\\'")
    slug_js = slug.replace("\\", "\\\\").replace("'", "\\'")
    bloque = """
<div id="exportctl___MID__" style="position:fixed;top:12px;right:12px;z-index:10000;
     display:flex;flex-direction:column;gap:6px;font-family:system-ui,Segoe UI,sans-serif">
  <button id="btndeck___MID__" style="cursor:pointer;border:1px solid #333;background:#fff;
     color:#222;padding:6px 10px;border-radius:6px;box-shadow:0 2px 6px rgba(0,0,0,.25);
     font-size:12px;font-weight:600">Exportar lamina (deck)</button>
</div>
<script>
(function(){
  var MAP_ID = '__MID__';
  var TITULO = '__TITULO__';
  var SLUG = '__SLUG__';
  var FUENTE = 'ITT Ciudad Paraiso - CDO Alcaldia de Santiago de Cali';
  var DORADO = '#8A6D1A';
  function fechaISO(){
    var d = new Date();
    var mm = ('0'+(d.getMonth()+1)).slice(-2);
    var dd = ('0'+d.getDate()).slice(-2);
    return d.getFullYear()+'-'+mm+'-'+dd;
  }
  function getContainer(){
    return document.getElementById(MAP_ID);
  }
  // Paso (a): captura SOLO el contenedor del mapa a un canvas, ocultando antes
  // los botones de export, el selector de capas, los tiles base (CORS), el
  // panel flotante de graficas y el slider de intensidad del calor; restaura
  // todo despues. Devuelve el canvas via callback.
  function capturarMapa(cont, cb){
    var ctl = document.getElementById('exportctl_'+MAP_ID);
    if(ctl) ctl.style.display = 'none';
    var layerCtls = cont.querySelectorAll('.leaflet-control-layers');
    layerCtls.forEach(function(lc){ lc.dataset.prevDisplay = lc.style.display; lc.style.display = 'none'; });
    // Panel de graficas flotante (#seg-panel o #eu-panel) y slider del calor.
    var panel = cont.querySelector('#seg-panel, #eu-panel, #ipm-panel, #itt-panel');
    var prevPanelDisplay = null;
    if(panel){ prevPanelDisplay = panel.style.display; panel.style.display = 'none'; }
    var slider = cont.querySelector('#heat-intensity-ctrl');
    var prevSliderDisplay = null;
    if(slider){ prevSliderDisplay = slider.style.display; slider.style.display = 'none'; }
    // CORS: ocultar tiles base y aplicar fondo claro.
    var tilePanes = cont.querySelectorAll('.leaflet-tile-pane');
    var fondoPrev = cont.style.background;
    cont.style.background = '#F4F6F9';
    tilePanes.forEach(function(tp){ tp.dataset.prevOpacity = tp.style.opacity; tp.style.opacity = '0'; });
    function restaurar(){
      if(ctl) ctl.style.display = '';
      layerCtls.forEach(function(lc){ lc.style.display = ''; });
      if(panel) panel.style.display = (prevPanelDisplay || '');
      if(slider) slider.style.display = (prevSliderDisplay || '');
      tilePanes.forEach(function(tp){ tp.style.opacity = tp.dataset.prevOpacity || ''; });
      cont.style.background = fondoPrev;
    }
    html2canvas(cont, {useCORS:true, backgroundColor:'#F4F6F9', logging:false, scale:2})
      .then(function(canvas){ restaurar(); cb(null, canvas); })
      .catch(function(err){ restaurar(); cb(err, null); });
  }
  // Paso previo: asegura el panel #seg-panel/#eu-panel expandido (si usa
  // <details>) para que Chart.js tenga el canvas visible y pintado, luego
  // captura el canvas real del grafico a un dataURL PNG. Devuelve null si
  // no hay canvas o esta vacio. cb(dataUrl|null).
  function capturarGrafico(cb){
    var panel = document.querySelector('#seg-panel, #eu-panel, #ipm-panel, #itt-panel');
    if(!panel){ cb(null); return; }
    var dets = panel.querySelectorAll('details');
    dets.forEach(function(d){ d.setAttribute('open', 'open'); });
    // Pequeno tiempo para que el canvas quede visible y Chart.js pinte.
    setTimeout(function(){
      var cv = panel.querySelector('canvas[id*="chart"]') || panel.querySelector('canvas');
      if(!cv || !cv.width || !cv.height){ cb(null); return; }
      try { cb(cv.toDataURL('image/png')); }
      catch(e){ cb(null); }
    }, 250);
  }
  // Paso (b): construye el div-lamina temporal con estructura de deck.
  // La imagen del mapa (canvas de (a)) va en la columna derecha; el panel
  // existente (si lo hay) se clona en la columna izquierda.
  function construirLamina(mapaDataUrl, graficoDataUrl){
    var lam = document.createElement('div');
    lam.id = 'deck-lamina';
    lam.style.cssText = 'position:fixed;top:0;left:0;z-index:2147483647;'+
      'width:1200px;background:#FFFFFF;color:#333;'+
      'font-family:system-ui,Segoe UI,sans-serif;'+
      'box-sizing:border-box;overflow:hidden;';
    // ENCABEZADO (barra con titulo + linea de seccion gris + borde dorado).
    var head = document.createElement('div');
    head.style.cssText = 'padding:16px 22px 12px 22px;border-bottom:3px solid '+DORADO+';';
    head.innerHTML =
      '<div style="font-size:20px;font-weight:700;color:#222;line-height:1.2">'+TITULO+'</div>'+
      '<div style="font-size:12px;color:#888;margin-top:4px">ITT Ciudad Paraiso - Lamina de deck (PMU)</div>';
    lam.appendChild(head);
    // CUERPO en 2 columnas (grid).
    var body = document.createElement('div');
    body.style.cssText = 'display:grid;grid-template-columns:35% 65%;gap:16px;'+
      'padding:16px 22px;align-items:start;';
    // Columna izquierda: resumen / cifras (clon del panel si existe).
    var colIzq = document.createElement('div');
    colIzq.style.cssText = 'min-width:0;';
    var panelOrig = document.querySelector('#seg-panel, #eu-panel, #ipm-panel, #itt-panel');
    if(panelOrig){
      var clon = panelOrig.cloneNode(true);
      clon.removeAttribute('id');
      // Sacar el clon del flujo fijo para que fluya dentro de la columna.
      clon.style.position = 'static';
      clon.style.top = 'auto'; clon.style.left = 'auto';
      clon.style.right = 'auto'; clon.style.bottom = 'auto';
      clon.style.zIndex = 'auto';
      clon.style.maxWidth = 'none';
      clon.style.width = '100%';
      clon.style.boxShadow = 'none';
      // Expandir cualquier <details> para que la serie se vea.
      var dets = clon.querySelectorAll('details');
      dets.forEach(function(d){ d.setAttribute('open', 'open'); });
      // El canvas de Chart.js se clona VACIO (los canvas no conservan su
      // contenido al clonar). Lo sustituimos por un <img> con la imagen
      // capturada del canvas real para que el grafico SE VEA en la lamina.
      if(graficoDataUrl){
        var cvClon = clon.querySelector('canvas');
        if(cvClon){
          var imgChart = document.createElement('img');
          imgChart.src = graficoDataUrl;
          imgChart.style.cssText = 'display:block;width:100%;height:auto;';
          if(cvClon.parentNode){ cvClon.parentNode.replaceChild(imgChart, cvClon); }
        }
      }
      colIzq.appendChild(clon);
    } else {
      colIzq.innerHTML =
        '<div style="font-size:15px;font-weight:700;color:#222;margin-bottom:6px">'+TITULO+'</div>'+
        '<div style="font-size:13px;color:#666">Resumen no disponible</div>';
    }
    // Columna derecha: imagen del mapa escalada al ancho de la columna.
    var colDer = document.createElement('div');
    colDer.style.cssText = 'min-width:0;';
    var img = document.createElement('img');
    img.src = mapaDataUrl;
    img.style.cssText = 'display:block;width:100%;height:auto;'+
      'border:1px solid #d9dde3;border-radius:6px;';
    colDer.appendChild(img);
    body.appendChild(colIzq);
    body.appendChild(colDer);
    lam.appendChild(body);
    // PIE (corte + fuente + nota).
    var pie = document.createElement('div');
    pie.style.cssText = 'padding:10px 22px 14px 22px;border-top:1px solid #e2e5ea;'+
      'font-size:11px;color:#777;';
    pie.innerHTML = 'Corte: '+new Date().toLocaleString()+
      '  |  Fuente: '+FUENTE+'  |  Mapa ilustrativo';
    lam.appendChild(pie);
    document.body.appendChild(lam);
    return lam;
  }
  // Dialogo "Guardar como" (Chrome/Edge) con fallback a descarga clasica.
  function guardarPDF(blob, nombre){
    function descargaFallback(){
      var url = URL.createObjectURL(blob);
      var a = document.createElement('a');
      a.href = url; a.download = nombre;
      document.body.appendChild(a); a.click(); document.body.removeChild(a);
      setTimeout(function(){ URL.revokeObjectURL(url); }, 1500);
    }
    if(window.showSaveFilePicker){
      var tipos = [{description:'Documento PDF', accept:{'application/pdf':['.pdf']}}];
      window.showSaveFilePicker({suggestedName:nombre, types:tipos})
        .then(function(handle){ return handle.createWritable(); })
        .then(function(writable){ return writable.write(blob).then(function(){ return writable.close(); }); })
        .catch(function(e){
          // Si el usuario cancela (AbortError) no se descarga nada; otros -> fallback.
          if(e && e.name === 'AbortError'){ return; }
          descargaFallback();
        });
    } else {
      descargaFallback();
    }
  }
  // Bounds del poligono de la zona de estudio (Ciudad Paraiso) para
  // reencuadrar el mapa SOLO en la generacion de la lamina (no afecta la
  // vista interactiva previa del usuario; tras exportar queda en este
  // encuadre pero la interactividad se mantiene).
  var ZONA_BOUNDS = L.latLngBounds([[3.442445,-76.53259],[3.44791,-76.526004]]);
  function exportar(){
    var cont = getContainer();
    if(!cont){ alert('No se encontro el contenedor del mapa.'); return; }
    // 1) Forzar el panel a 'open' y capturar el grafico real a dataURL.
    capturarGrafico(function(graficoDataUrl){
      var mp = window[MAP_ID];
      // 2) fitBounds al poligono. maxZoom ENTERO evita el zoom fraccional que
      //    descoloca las capas en html2canvas (desfase al imprimir).
      var capturado = false;
      function trasEncuadre(){
        if(capturado){ return; }
        capturado = true;
        try { if(mp){ mp.invalidateSize(false); } } catch(e){}
        // 3) Margen corto tras asentar panes/tiles, luego capturar.
        setTimeout(function(){
        // 4) Capturar el mapa (ocultando botones/tiles/panel/slider).
        capturarMapa(cont, function(err, canvasMapa){
          if(err){ alert('Error al capturar el mapa: '+err); return; }
          var mapaDataUrl = canvasMapa.toDataURL('image/jpeg', 0.95);
          // 5) Construir lamina (cifras + grafico como imagen + mapa).
          var lam = construirLamina(mapaDataUrl, graficoDataUrl);
          function limpiar(){
            if(lam && lam.parentNode) lam.parentNode.removeChild(lam);
          }
          // 6) Capturar la lamina y generar el PDF.
          html2canvas(lam, {backgroundColor:'#FFFFFF', logging:false, scale:2, useCORS:true})
            .then(function(canvas){
              limpiar();
              var imgData = canvas.toDataURL('image/jpeg', 0.95);
              var jsPDFns = (window.jspdf && window.jspdf.jsPDF) ? window.jspdf.jsPDF : window.jsPDF;
              var w = canvas.width, hh = canvas.height;
              var orient = (w >= hh) ? 'landscape' : 'portrait';
              var doc = new jsPDFns({orientation:orient, unit:'pt', format:[w, hh]});
              doc.addImage(imgData, 'JPEG', 0, 0, w, hh);
              var blob = doc.output('blob');
              var nombre = 'deck_'+SLUG+'_'+fechaISO()+'.pdf';
              guardarPDF(blob, nombre);
            })
            .catch(function(e){
              limpiar();
              alert('Error al exportar la lamina: '+e);
            });
        });
        }, 350);
      }
      // Disparar fitBounds y esperar 'moveend'; fallback por timeout si no llega.
      try {
        if(mp){
          mp.once('moveend', trasEncuadre);
          mp.fitBounds(ZONA_BOUNDS, {padding:[10,10], maxZoom: 17, animate: false});
          setTimeout(trasEncuadre, 1200);
        } else {
          trasEncuadre();
        }
      } catch(e){ trasEncuadre(); }
    });
  }
  function init(){
    var b = document.getElementById('btndeck_'+MAP_ID);
    if(b) b.addEventListener('click', function(){ exportar(); });
  }
  if(document.readyState === 'loading'){
    document.addEventListener('DOMContentLoaded', init);
  } else { init(); }
})();
</script>
"""
    bloque = (bloque.replace("__MID__", map_id)
                    .replace("__TITULO__", titulo_js)
                    .replace("__SLUG__", slug_js))
    mapa.get_root().html.add_child(folium.Element(bloque))
    return mapa

## 6. Construccion del mapa (Folium) - estandar Ciudad Paraiso

In [ ]:
ZOOM_START = 15
ZOOM_START = 16
# Mapa ITT parcial: estandar Ciudad Paraiso. Limite azul fijo, manzanas base,
# coropleta ITT por nivel (solo dimension social poblada).
import folium
_c = POLIGONO_ZONA.geometry.union_all().centroid
m = folium.Map(location=[_c.y,_c.x], zoom_start=ZOOM_START, tiles=None, control_scale=True,
               zoom_snap=0.25, zoom_delta=0.1, max_zoom=22)
m.get_root().width="100%"; m.get_root().height="100%"
m.get_root().header.add_child(folium.Element(
    "<style>html,body{height:100%;margin:0;padding:0}"
    ".folium-map{position:absolute;top:0;bottom:0;left:0;right:0;width:100%;height:100%}"
    ".leaflet-top,.leaflet-bottom{z-index:1000}"
    "#itt-panel{z-index:1001;background:rgba(255,255,255,.95);backdrop-filter:blur(4px);"
    "-webkit-backdrop-filter:blur(4px);box-shadow:0 4px 15px rgba(0,0,0,.2)}"
    ".export-ctl{z-index:1002}</style>"))
folium.TileLayer(tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
    attr="Google", name="Google Streets", show=True, max_zoom=22, max_native_zoom=20).add_to(m)
folium.TileLayer(tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Esri Satelite", show=False, max_zoom=22, max_native_zoom=20).add_to(m)
folium.TileLayer(tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="Google", name="Google Satelite", show=False, max_zoom=22, max_native_zoom=20).add_to(m)
# Mascara oscura fuera de la zona.
_zg = POLIGONO_ZONA.geometry.union_all()
_mundo=[[-180,-85],[180,-85],[180,85],[-180,85],[-180,-85]]; _ag=[]
for _gm in (list(_zg.geoms) if _zg.geom_type=="MultiPolygon" else [_zg]):
    _ag.append([[x,y] for x,y in _gm.exterior.coords])
folium.GeoJson({"type":"Feature","properties":{},"geometry":{"type":"Polygon","coordinates":[_mundo]+_ag}},
    name="Mascara (foco en la zona)", control=False,
    style_function=lambda f:{"fillColor":"#000","color":"#000","weight":0,"fillOpacity":0.18}).add_to(m)

# 1) Coropleta ITT por manzana (nivel segun dimension social). Activa al inicio.
def _popup_itt(r):
    if bool(r["sin_pob"]):
        return '<div style="font-family:system-ui;font-size:12px;background:%s;padding:6px 8px;border-radius:6px"><b>Manzana sin poblacion</b></div>' % BG
    v=float(r["itt_parcial"]); nom,_=_itt_nivel(v)
    return ('<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;background:%s;'
            'padding:7px 9px;border-radius:6px;min-width:210px">'
            '<div style="font-weight:700;color:%s">ITT parcial (base social)</div>'
            '<div>Score: <b>%.1f</b> / 100 &rarr; <b>%s</b></div>'
            '<div style="margin-top:3px;font-size:11px">Dimensiones:</div>'
            '<div style="font-size:11px">&bull; Social/Vivienda: <b>%.1f</b> (IPM %.1f%%)</div>'
            '<div style="font-size:11px;color:#999">&bull; Seguridad / Convivencia / Movilidad / Entorno: sin dato</div>'
            '<div style="font-size:9px;color:#888;margin-top:4px">ITT PARCIAL: solo base social. Otras dimensiones pendientes (DATIC).</div></div>'
            % (BG,LIMITE,v,nom,v,float(r["ipm"])))
_fg_itt = folium.FeatureGroup(name="ITT: manzanas (nivel por base social)", show=True)
for _,r in mz.iterrows():
    if bool(r["sin_pob"]):
        _col="#bdbdbd"; _tt="Sin poblacion"
    else:
        _nom,_col=_itt_nivel(float(r["itt_parcial"])); _tt="ITT parcial %.1f (%s)"%(float(r["itt_parcial"]),_nom)
    folium.GeoJson(r["geometry"].__geo_interface__,
        style_function=(lambda f,c=_col:{"fillColor":c,"color":"#ffffff","weight":0.6,"fillOpacity":0.82}),
        tooltip=_tt, popup=folium.Popup(_popup_itt(r),max_width=270)).add_to(_fg_itt)
_fg_itt.add_to(m)

# 2) Capa base de manzanas (relleno pastel, borde gris, tooltip con ID y area).
def _tt_mz(r):
    _id = r.get("COD_MZN","") or r.get("OID_","")
    _area = ("{:,.0f}".format(float(r["area_m2"]))).replace(",",".")
    return ("ID manzana: %s<br>Barrio: Barrio Obrero (Comuna 9)<br>Area: %s m2" % (_id,_area))
_fg_base = folium.FeatureGroup(name="Manzanas (base catastral)", show=False)
for _,r in mz.iterrows():
    folium.GeoJson(r["geometry"].__geo_interface__,
        style_function=lambda f:{"fillColor":"#cfe3f2","color":OKI_GRIS,"weight":1.0,"fillOpacity":0.38},
        tooltip=_tt_mz(r)).add_to(_fg_base)
_fg_base.add_to(m)

# 3) Capas de dimensiones SIN DATO (placeholders vacios, informativos en el control).
for _nomcapa in ["Seguridad (sin datos Comuna 9)","Convivencia (sin datos Comuna 9)",
                 "Movilidad (sin datos Comuna 9)","Entorno Urbano (sin datos Comuna 9)"]:
    folium.FeatureGroup(name=_nomcapa, show=False).add_to(m)

# 4) Limite perimetral de la zona (azul, fijo y visible).
folium.GeoJson(POLIGONO_ZONA.to_json(), name="Limite Barrio Obrero", show=True,
    style_function=lambda f:{"color":LIMITE,"weight":2.5,"fill":False}).add_to(m)

# Leyenda de niveles ITT.
_items=""
for lo,hi,nom,col in ITT_NIVELES:
    _items+=('<div style="display:flex;align-items:center;gap:6px;margin:1px 0">'
             '<i style="width:16px;height:12px;background:%s;display:inline-block;border:1px solid #999"></i>'
             '<span style="font-size:10px">%s (%d-%d)</span></div>' % (col,nom,int(lo),int(hi)))
_items+=('<div style="display:flex;align-items:center;gap:6px;margin:1px 0">'
         '<i style="width:16px;height:12px;background:#bdbdbd;display:inline-block;border:1px solid #999"></i>'
         '<span style="font-size:10px">Sin poblacion / sin dato</span></div>')
_leyenda=('<div id="itt-leyenda" style="position:fixed;bottom:24px;right:24px;z-index:1000;'
          'background:'+BG+';padding:8px 10px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,.25);'
          'font-family:system-ui,Segoe UI,sans-serif;color:'+OKI_GRIS+';max-width:230px">'
          '<div style="font-weight:700;font-size:11px;margin-bottom:4px">Nivel ITT (base social)</div>'
          +_items+
          '<div style="font-size:9px;color:#888;margin-top:4px">ITT parcial: solo dimension social (100 - IPM).</div></div>')
m.get_root().html.add_child(folium.Element(_leyenda))
m.fit_bounds(BO_BOUNDS, padding=(30,30))
folium.LayerControl(collapsed=True, position="topright").add_to(m)
print("Mapa ITT parcial construido. Manzanas:", len(mz))

## 7. Widget flotante: score ITT parcial + 5 dimensiones + cruce IPM

In [ ]:
SUB_PANEL = "(oficial)"
SUB_PANEL = "(V1)"
# Widget flotante #itt-panel: score ITT parcial de zona, distribucion por nivel,
# estado de las 5 dimensiones (social con dato, 4 sin dato) y cruce con IPM.
_cont={nom:0 for _lo,_hi,nom,_c in ITT_NIVELES}
for _v in mz.loc[~mz["sin_pob"],"itt_parcial"]:
    _n,_=_itt_nivel(_v); _cont[_n]+=1
# Serie de dimensiones para el grafico (social con valor; resto en 0 marcado sin dato).
_dim_labels=["Social/Vivienda","Seguridad","Convivencia","Movilidad","Entorno Urb."]
_dim_vals=[ITT_ZONA,0,0,0,0]
_dim_hay=[True,False,False,False,False]
_chart=json.dumps({"labels":_dim_labels,"data":_dim_vals,"hay":_dim_hay}, ensure_ascii=False)

_resumen=('<div style="margin:2px 0 6px 0">ITT parcial de zona (base social): '
          '<b style="font-size:15px;color:%s">%.1f</b> / 100 &nbsp;&rarr;&nbsp; <b>%s</b></div>'
          % (LIMITE,ITT_ZONA,_niv_zona))
_resumen+='<div style="font-size:10px;color:#666;margin-bottom:6px">IPM promedio de zona: %.1f%% de personas. Manzanas con poblacion: %d.</div>' % (IPM_ZONA,int((~mz["sin_pob"]).sum()))
_resumen+='<div style="margin:4px 0 2px;font-weight:700;font-size:11px">Manzanas por nivel ITT:</div>'
for _lo,_hi,nom,col in ITT_NIVELES:
    _resumen+=('<div style="display:flex;align-items:center;gap:6px;margin:1px 0;font-size:11px">'
               '<i style="width:12px;height:12px;background:%s;display:inline-block;border-radius:2px;border:1px solid #777"></i>'
               '<span>%s: <b>%d</b></span></div>' % (col,nom,_cont[nom]))

_panel = r"""
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<div id="itt-panel" style="position:fixed;bottom:25px;left:20px;z-index:1001;
     max-width:400px;width:min(400px,44vw);max-height:84vh;overflow-y:auto;
     padding:11px 13px;border-radius:10px;font-family:system-ui,Segoe UI,sans-serif;font-size:12px;color:__GRIS__">
  <div style="display:flex;align-items:center;justify-content:space-between;gap:8px">
    <div style="font-weight:700;font-size:13px">ITT parcial - Barrio Obrero __SUB__</div>
    <button id="itt-toggle" title="Minimizar / expandir" style="cursor:pointer;border:1px solid #bbb;background:#fff;color:#333;width:24px;height:22px;border-radius:5px;font-size:13px;line-height:1;font-weight:700;flex:0 0 auto">_</button>
  </div>
  <div id="itt-body">
    <div style="background:#f0e9e0;border:1px dashed #c8a96a;border-radius:6px;padding:7px 9px;margin:6px 0;font-size:10px;color:#7a6a4a">
      <b>ITT PARCIAL.</b> Solo la dimension <b>Social/Vivienda</b> (derivada del IPM) tiene dato
      para Barrio Obrero (Comuna 9). Las otras 4 estan <b>Sin datos para Comuna 9 (pendiente DATIC)</b>.
    </div>
    __RESUMEN__
    <details open style="margin-top:8px">
      <summary style="cursor:pointer;font-weight:600;color:__LIM__;outline:none">Dimensiones del ITT / Ocultar</summary>
      <div style="position:relative;height:200px;width:100%;margin-top:6px"><canvas id="itt-chart"></canvas></div>
      <div style="font-size:9px;color:#777;margin-top:3px">Escala 0-100. Barras grises = dimension sin dato (pendiente DATIC).</div>
    </details>
    <details style="margin-top:6px">
      <summary style="cursor:pointer;font-weight:600;color:#999;outline:none">Que falta para el ITT completo / Ver</summary>
      <div style="font-size:11px;color:#555;margin-top:6px;line-height:1.4">
        Para completar el ITT de Comuna 9 se necesitan, georreferenciados en Barrio Obrero:
        homicidios/hurtos/VIF (Seguridad), comparendos por rinas y SPA (Convivencia),
        siniestros viales y red peatonal (Movilidad) y censo arboreo/NDVI (Entorno Urbano).
        El dataset ITT actual cubre Ciudad Paraiso (Comuna 3).
      </div>
    </details>
  </div>
</div>
<script>
(function(){
  var D=__CHART__;
  var VERDE="#009E73", GRIS="#c9c9c9";
  function dibujar(){
    var cv=document.getElementById("itt-chart"); if(!cv||typeof Chart==="undefined") return;
    var cols=D.hay.map(function(h){ return h?VERDE:GRIS; });
    new Chart(cv.getContext("2d"),{type:"bar",
      data:{labels:D.labels,datasets:[{data:D.data.map(function(v,i){return D.hay[i]?v:0;}),backgroundColor:cols,borderWidth:1}]},
      options:{indexAxis:"y",responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false},
        tooltip:{callbacks:{label:function(c){return D.hay[c.dataIndex]?("Score: "+c.parsed.x):"Sin dato (pendiente DATIC)";}}}},
        scales:{x:{beginAtZero:true,max:100,ticks:{font:{size:9}},title:{display:true,text:"Cantidad",font:{size:9}}},y:{ticks:{font:{size:9}}}}}});
  }
  function initColapso(){
    var btn=document.getElementById("itt-toggle"), body=document.getElementById("itt-body");
    if(!btn||!body) return;
    btn.addEventListener("click",function(){ var o=(body.style.display==="none"); body.style.display=o?"":"none"; btn.textContent=o?"_":"+"; });
  }
  if(document.readyState==="complete"){ setTimeout(dibujar,300); initColapso(); }
  else { window.addEventListener("load",function(){ setTimeout(dibujar,300); initColapso(); }); }
})();
</script>
"""
_panel=(_panel.replace("__GRIS__",OKI_GRIS).replace("__LIM__",LIMITE).replace("__SUB__",SUB_PANEL)
             .replace("__RESUMEN__",_resumen).replace("__CHART__",_chart))
m.get_root().html.add_child(folium.Element(_panel))
print("Widget ITT parcial inyectado.")

## 8. Exportacion del HTML

In [ ]:
agregar_export(m, "ITT parcial (base social) - Barrio Obrero (oficial, 45 manzanas)", "itt_bo_original")
html=os.path.join(MAPAS_DIR,"mapa_itt_barrio_obrero_original.html")
m.save(html)
print("Mapa ITT Barrio Obrero (oficial, 45 manzanas) exportado:", html)
m